In [ ]:
# CSMODEL S01
# G25 - Monty Python and the Holy Bell Curve 
# Members: 
# Abogadie, Nina 
# De Gracia, Kaleela 
# Domingo, Stefan
# Jose, Yesha
# Perez, Jose 

# CSMODEL MCO1 
### Competitive Games and its Effect on Strategic Decision Making 

#### Dataset Description:
##### TFT(League of Legends) - High Elo Ranked Games  
Link: https://www.kaggle.com/datasets/gyejr95/tft-match-data 

Description: 
It is a data set that builds 10,000 matches of challengers, so you can see the rank of each user in each game and what pattern each rank has.

##### PUBG Match Deaths and Statistics     
Link: https://www.kaggle.com/datasets/skihikingkevin/pubg-match-deaths 

Description:
In this Kaggle Dataset, I provide over 720,000 competitive matches from the popular game PlayerUnknown's Battlegrounds. 

# Importing python libraries to be used 
Pandas - main library used for creating and manipulating dataframes. 

Numpy -  used for statistical calculations. 

Seaborn - data visualization. 

Abstract Syntax Trees - for treating strings like dicts and parsing their keys and values

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import ast
#setting the size of figures to be displayed in the notebook
sns.set_theme(rc={"figure.figsize": (12, 6)})

# Loading datasets to be used

In [ ]:
# Loading TFT datasets
TFT_C_Data_df = pd.read_csv("Datasets/TFT_Match_Data/TFT_Challenger_MatchData.csv")
TFT_D_Data_df = pd.read_csv("Datasets/TFT_Match_Data/TFT_Diamond_MatchData.csv")
TFT_GM_Data_df = pd.read_csv("Datasets/TFT_Match_Data/TFT_GrandMaster_MatchData.csv")
TFT_M_Data_df = pd.read_csv("Datasets/TFT_Match_Data/TFT_Master_MatchData.csv")
TFT_P_Data_df = pd.read_csv("Datasets/TFT_Match_Data/TFT_Platinum_MatchData.csv")

In [ ]:
# Loading PUBG datasets 
PUBG_death1_df = pd.read_csv("Datasets/PUBG_Match_Data/deaths/kill_match_stats_final_0.csv")
PUBG_death2_df = pd.read_csv("Datasets/PUBG_Match_Data/deaths/kill_match_stats_final_1.csv")
PUBG_death3_df = pd.read_csv("Datasets/PUBG_Match_Data/deaths/kill_match_stats_final_2.csv")
PUBG_agg_stats_df = pd.read_csv("Datasets/PUBG_Match_Data/aggregate/agg_match_stats_4.csv")

# Data Preprocessing for TFT


For TFT the biggest question we want answered is if champion and item choice differs drastically between low and high ranking players.

Identifying how certain champions are picked over others, if there are any, may give key insight into how low and high skill players make decisions whose consequences stretch far into the game. In effect we can sort of view it as analogous to how one decides to move their pieces in a chess game which makes sense given TFT is an AutoChess game.

We first process the dataset by dropping unnecessary columns. "gameId" is metadata that isn't really relevant to statistical analysis and "gameDuration" takes into account things such as loading times which doesn't really affect gameplay.

In [ ]:
TFT_C_Processed = TFT_C_Data_df.drop(columns=['gameId','gameDuration'])
TFT_D_Processed = TFT_D_Data_df.drop(columns=['gameId','gameDuration'])
TFT_GM_Processed = TFT_GM_Data_df.drop(columns=['gameId','gameDuration'])
TFT_M_Processed = TFT_M_Data_df.drop(columns=['gameId','gameDuration'])
TFT_P_Processed = TFT_P_Data_df.drop(columns=['gameId','gameDuration'])

TFT_C_Processed.head()

Next we convert "inGameDuration" to be measured in minutes

In [ ]:

TFT_C_Processed['ingameDuration'] = TFT_C_Processed['ingameDuration'] / 60
TFT_C_Processed['ingameDuration'] = np.round(TFT_C_Processed['ingameDuration'], decimals=2)

TFT_D_Processed['ingameDuration'] = TFT_D_Processed['ingameDuration'] / 60
TFT_D_Processed['ingameDuration'] = np.round(TFT_D_Processed['ingameDuration'], decimals=2)

TFT_GM_Processed['ingameDuration'] = TFT_GM_Processed['ingameDuration'] / 60
TFT_GM_Processed['ingameDuration'] = np.round(TFT_GM_Processed['ingameDuration'], decimals=2)

TFT_M_Processed['ingameDuration'] = TFT_M_Processed['ingameDuration'] / 60
TFT_M_Processed['ingameDuration'] = np.round(TFT_M_Processed['ingameDuration'], decimals=2)

TFT_P_Processed['ingameDuration'] = TFT_P_Processed['ingameDuration'] / 60
TFT_P_Processed['ingameDuration'] = np.round(TFT_P_Processed['ingameDuration'], decimals=2)



Now we move on to processing the champion data. Notice below how data is encoded similar to a dict where champions are treated as keys

In [ ]:
TFT_C_Processed['champion'].head(5)

We can therefore use the ast library to extract the champion names and items.

In [ ]:
champ_list_C = TFT_C_Processed['champion'].apply(lambda x: list(ast.literal_eval(x).keys()))
champ_list_D = TFT_D_Processed['champion'].apply(lambda x: list(ast.literal_eval(x).keys()))
champ_list_GM = TFT_GM_Processed['champion'].apply(lambda x: list(ast.literal_eval(x).keys()))
champ_list_M = TFT_M_Processed['champion'].apply(lambda x: list(ast.literal_eval(x).keys()))
champ_list_P = TFT_P_Processed['champion'].apply(lambda x: list(ast.literal_eval(x).keys()))

combo_list_C = TFT_C_Processed['combination'].apply(lambda x: list(ast.literal_eval(x).keys()))
combo_list_D = TFT_D_Processed['combination'].apply(lambda x: list(ast.literal_eval(x).keys()))
combo_list_GM = TFT_GM_Processed['combination'].apply(lambda x: list(ast.literal_eval(x).keys()))
combo_list_M = TFT_M_Processed['combination'].apply(lambda x: list(ast.literal_eval(x).keys()))
combo_list_P = TFT_P_Processed['combination'].apply(lambda x: list(ast.literal_eval(x).keys()))

print(combo_list_C.head(1))
print(champ_list_C.head(1))


We now have strings of champion and item names, we can then just parse the strings and get the counts of each champion and item

In [ ]:
champ_counts_C = champ_list_C.explode().value_counts()
champ_counts_D = champ_list_D.explode().value_counts()
champ_counts_GM = champ_list_GM.explode().value_counts()
champ_counts_M = champ_list_M.explode().value_counts()
champ_counts_P = champ_list_P.explode().value_counts()

combo_counts_C = combo_list_C.explode().value_counts()
combo_counts_D = combo_list_D.explode().value_counts()
combo_counts_GM = combo_list_GM.explode().value_counts()
combo_counts_M = combo_list_M.explode().value_counts()
combo_counts_P = combo_list_P.explode().value_counts()


# Exploratory Data Analysis for TFT

## 


To determine which champions are often played together, a heatmap is created for every rank to visualize the choices high elo players make in team composition.


In [ ]:
#to check what champions are often played together

champ_matrix_D = champ_list_D.apply(lambda x: '|'.join(x)).str.get_dummies()
correlation_matrix = champ_matrix_D.corr()
sns.heatmap(correlation_matrix)


In [ ]:
#to check what champions are often played together

champ_matrix_P = champ_list_P.apply(lambda x: '|'.join(x)).str.get_dummies()
correlation_matrix = champ_matrix_P.corr()
sns.heatmap(correlation_matrix)


In [ ]:
#to check what champions are often played together

champ_matrix_GM = champ_list_GM.apply(lambda x: '|'.join(x)).str.get_dummies()
correlation_matrix = champ_matrix_GM.corr()
sns.heatmap(correlation_matrix)


In [ ]:
#to check what champions are often played together

champ_matrix_C = champ_list_C.apply(lambda x: '|'.join(x)).str.get_dummies()
correlation_matrix = champ_matrix_C.corr()
sns.heatmap(correlation_matrix)


From the visualization, Platinum and Diamond players tend to have more variety in champion selections compared to their higher elo counterparts.

# Data Preprocessing for PUBG

### Preprocessing aggregate match data

As a disclaimer, the PUBG dataset we are using gives the statistics for organizes its data into chunks of 5 categorized as the death statistics and the aggregate statistics. For the aggregate statistics 4 of the csv files exceeded the Git LFS file size limit Github allowed hence why only a size csv file is being used for exploring aggregate data. 

Aggregation data for PUBG matches includes match metadata such as teamIDs, team rankings, kills, survival time, etc. The baseline metric we will be using to measure success is a team's placement at the end of a match.

Our main questions then pertain to how certain factors like survival time and total kills correlate to team placement, additionally since squad size is also recorded we can investigate how collaboration and the lack thereof correlates with success/team placement. 

Beginning preprocessing we drop all rows containing null values as well as columns that we won't be needing for analysis. 

In [ ]:
PUBG_agg_stats_df = PUBG_agg_stats_df.dropna()
PUBG_agg_stats_df = PUBG_agg_stats_df.drop(columns=['date','game_size','match_id','match_mode','team_id','player_name'])

Next we convert the "player_survive_time" column to be measured in minutes rather than seconds.

In [ ]:
PUBG_agg_stats_df['player_survive_time'] = PUBG_agg_stats_df['player_survive_time']/60
PUBG_agg_stats_df['player_survive_time'] = np.round(PUBG_agg_stats_df['player_survive_time'], decimals=2)

Now we verify the values of the survival time column. As shown below, there are several extremely high outlier values within the "player_survive_time" column. We attributed these values as encoding errors as they are statistically impossible.

In [ ]:
print(PUBG_agg_stats_df['player_survive_time'].sort_values(ascending=False).head(10))

To address this we decided to drop these rows entirely based on the 99th percentile value

In [ ]:
max = PUBG_agg_stats_df['player_survive_time'].quantile(0.99) #this returns 32.24 minutes which is a reasonable number as the average PUBG match does last around 20-30 minutes
PUBG_agg_stats_df = PUBG_agg_stats_df[PUBG_agg_stats_df['player_survive_time'] < max]

Next we verify the team placements to ensure there are no odd values and sure enough there are. Team placement in PUBG only goes as low as 1 as that indicates they are the winner of the match, therefore having a placement of 0 is impossible. Thus these rows have to be removed.

In [ ]:
print(PUBG_agg_stats_df['team_placement'].sort_values().head(10))

In [ ]:
PUBG_agg_stats_df = PUBG_agg_stats_df.query('team_placement != 0')

### Preprocessing death match data


Another disclaimer before proceeding, as mentioned at the beginning of the preprocessing of the aggregate data, the death data was split across 5 differing csv files. All files combined totaled to over 10GB of data which became unwieldy to process as we frequently ran into MemoryErrors due to hardware limitations. As such we have chosen to only process 3 of the original 5 files.

Death data for PUBG includes data such as the killer and victim's positions on the map, what the cause of death was, who the killer and victim were, etc.

Since the death match data is split across multiple files our first step in preprocessing is combining all the dataframes into one big dataframe. All the files are formatted the exact same so we can use the pandas concat function to quickly stack them ontop of each other.

In [ ]:
PUBG_death_df = pd.concat([PUBG_death1_df,PUBG_death2_df,PUBG_death3_df])

Next we proceed by dropping rows that contain empty values which are quite numerous since players dying via in game mechanics like drowning or being outside the safe zone leave columns 2-5 empty as there is no player data to encode as the killer. We then also drop other columns we don't need such as match metadata and player names.

In [ ]:
PUBG_death_df = PUBG_death_df.dropna()
PUBG_death_df = PUBG_death_df.drop(columns=['killer_name','map','match_id','victim_name','victim_placement'])

Next we create a new column that calculates the distance between a killer and their victim. We can use the euclidean distance formula since we are given x and y coordinates. We shall also round this resulting distance to 2 decimal places for the sake of readability.

In [ ]:
dist = np.round(np.sqrt((PUBG_death_df['killer_position_x'] - PUBG_death_df['victim_position_x'])**2 + 
                        (PUBG_death_df['killer_position_y'] - PUBG_death_df['victim_position_y'])**2), 
                        decimals=2)

Before proceeding we first inspect the data within the new column and see that some kills that are encoded are seemingly impossible as they span farther than the actual map's length which is 800,000cm or 8km. When appending this series into the main dataframe we can see these kills are impossible, take for example killing someone with a punch over 8km away. Whether these are hackers, encoding errors, or the game misattributing a kill to the wrong player, these data points are unusable.

In [ ]:
PUBG_death_df['kill_distance'] = dist

PUBG_death_df[PUBG_death_df['kill_distance'] > 800000][
    ['killed_by','killer_position_x', 'killer_position_y', 'victim_position_x', 'victim_position_y','kill_distance']
].head(15)

Another oddity that we needed to consider is self kills or kills where the kill_distance is exactly 0.0. As seen below a majority of these are due to grenades which make sense as people can blow themselves up with their own grenades by accident. Much like the previous example above, these data points are not needed for our use case.

In [ ]:
PUBG_death_df[PUBG_death_df['kill_distance'] == 0][
    ['killed_by','killer_position_x', 'killer_position_y', 'victim_position_x', 'victim_position_y','kill_distance']
].head(15)

A lot of these outliers occur when either the victim or killer is positioned exactly at (0,0) therefore we will be removing all entries that contain those exact coordinates for either the killer or victim positions. We can safely use the (0,0) position as our indicator of a bad data point because due to the size of the player models it is almost impossible for a person to be standing at exact the (0,0) axis. While we're at it, we shall also remove all entries where the kill distance is exactly 0.

In [ ]:
PUBG_death_df = PUBG_death_df.query('killer_position_x != 0 and killer_position_y != 0 or victim_position_x != 0 and victim_position_y != 0')

In [ ]:
PUBG_death_df = PUBG_death_df.query('kill_distance != 0.0')

Now we can convert the data into meters which is simply applying a scalar divison by 100 to the entire column.

In [ ]:
PUBG_death_df['kill_distance'] = PUBG_death_df['kill_distance'] / 100
PUBG_death_df['kill_distance'] = np.round(PUBG_death_df['kill_distance'], decimals=2 )

Now we can do a final check through of the kill distance to see if any outliers remain. 

In [ ]:
PUBG_death_df['kill_distance'].describe()

We can still see some outliers remain in the data since the highest kill distance is 10,800.04m which is extremely unlikely as the longest ranged snipers in the game have an effective range of only 700m. We can do what we did earlier and cap the max kill entries based on a quantile value.

In [ ]:
max = PUBG_death_df['kill_distance'].quantile(0.97) # this gives us a distance of 505.78m which is an acceptable max range.
PUBG_death_df = PUBG_death_df[PUBG_death_df['kill_distance'] < max]

With the kill distance calculated and cleaned we can then safely drop the positional data as we won't be needing it any longer.

In [ ]:
PUBG_death_df = PUBG_death_df.drop(columns=['killer_position_x','killer_position_y','victim_position_x','victim_position_y'])

Now we can also convert the time into minutes

In [ ]:
PUBG_death_df['time'] = PUBG_death_df['time'] / 60
PUBG_death_df['time'] = np.round(PUBG_death_df['time'], decimals=2 )

# Exploratory Data Analysis for PUBG 

### Exploratory Data Analysis for aggregate match data

Our first relationship to investigate is if there exists a correlation between party size (solo, duo, squad) and the survival time of a player i.e. does playing with/without people correlate to how long they last in a game?

In [ ]:
print(PUBG_agg_stats_df['player_survive_time'].corr(PUBG_agg_stats_df['party_size']))

As indicated by the correlation calculation above, there exists an extremely weak correlation between the two, graphing them into a box plot we can investigate the relationship more clearly.

In [ ]:
sns.boxplot(data=PUBG_agg_stats_df, x='party_size', y='player_survive_time')

The visualization above explains the relationship more clearly, all party sizes share the same relative variance with their upper and lower whiskers relatively similar. It is in the upper quartile where we see the largest difference with sqauds having the largest range indicating the players on average survive longer when playing in a team of 4 as opposed to playing with a partner or solo. 

We can also estimate that the average survival time of a player regardless of team size lies somewhere between 10-15 minutes. This is corroborated by the calculated mean below.

In [ ]:
print(np.round(np.mean(PUBG_agg_stats_df['player_survive_time']), decimals=2))

Moving forward we shall look into how team placement correlates with other variables instead. Beginning with party size again. We use the same pearson correlation formula and find a moderate negative correlation between the two. 

In [ ]:
print(PUBG_agg_stats_df['team_placement'].corr(PUBG_agg_stats_df['party_size']))

This indicates that the bigger the squad size the lower that team ranks. This is a good observation as it indicates that playing with other people could help in placing higher on the leaderboard by the end of the game.

In [ ]:
sns.boxplot(data=PUBG_agg_stats_df, x='party_size', y='team_placement')

The box plot above shows an even clearer picture of this correlation. We can see that even though each size's lower whiskers are identical, meaning all team sizes have achieved first place, it is in their upper whiskers where we can see the largest difference. Solo players are almost always the first to die e.g. obtain a higher team placement while duos at worst reach the top 50s with squads being the most performant as the lowest a team of had placed being somewhere in the top 40. Overall what this graph shows us is that while success can be achieved by any team, it is the squads who manage to place in the top 20 on average.

Next we shall investigate how aggressive the top 20 placing teams are via measuring their average damage dealt and kills. To do so we shall be passing on a smaller dataframe which only contains teams who placed within the top 20, we then use this the pivot table function in order to calculate the average damage per combination of factors e.g. what is the average damage dealt by duos place within 2nd place?

We then plot this dataframe into a heatmap and below is the resulting plot.

In [ ]:
top = PUBG_agg_stats_df[PUBG_agg_stats_df['team_placement'] <= 20]
heat = top.pivot_table(index='party_size',columns='team_placement',values='player_dmg',aggfunc='mean')

sns.heatmap(heat, annot=True, fmt='.0f')

Based on the heatmap above, aggressive play is a highly effective strategy. We can clearly first place averages a significant jump in damage compared to the incremental damage increase we see from 20th to 2nd place.

Finally we investigate how collaborative the top 20 teams via the "player_dbno" value. In PUBG, if you are playing in duos or squads, you are not immediately eliminated when reaching 0 health. Instead you enter a "Down But Not Out" state where you are unable to shoot or melee but can crawl around and be rescued by a teammate. The "player_dbno" value tracks how many times a player has entered said state. 

The general idea behind investigating this relationship is we want to see on average how many times a player who was in a team that made it in the top 20 was downed and subsequently saved by another teammate indicating collaboration and teamwork. Of course we shall be ignoring all solo top placers for this.

In [ ]:
top = PUBG_agg_stats_df[PUBG_agg_stats_df['team_placement'] <= 20]
heat = top.pivot_table(index='party_size',columns='team_placement',values='player_dbno',aggfunc='mean')

sns.heatmap(heat, annot=True, fmt='.01f')

As indicated by the plot above, higher ranking teams average more "dbno's" which indicates that teammates are consistently looking after and rescuing each other from failed engagements. This shows that collaboration coupled with aggression from the previous plot are part of a winning strategy.

### Exploratory Data Analysis for death match data

The first thing we would like to investigate regarding death data is the most common causes of death within PUBG. To do so we take the 10 most frequently occuring values from the "killed_by" column and plot them into a histogram.

In [ ]:
top10 = PUBG_death_df['killed_by'].value_counts().nlargest(10).index

filtered10 = PUBG_death_df[PUBG_death_df['killed_by'].isin(top10)]

sns.countplot(filtered10 , x='killed_by', order=top10)

For reference the scientific notation located at the top left "1e6" is the scale with which count is being rendered at so these numbers are in intervals in the millions. The most common cause of death is "Down and Out" which is not a weapon more so death caused by being in the "Down But Not Out" state that was discussed earlier so we can safely disregard it. That being the case, the most common deaths are caused by midrange rifle weapons like the M416 and SCAR-L.

Next we'll be looking at kill distance to determine how far away killers are from their victims. Ideally we should see most kills happen within 100m.

In [ ]:
sns.histplot(PUBG_death_df['kill_distance'], bins=50, kde=True)

This corroborates with the previous plot, we see most kills happen within 50 meters and skewing towards closer range kills. This indicates most engagements happening in close ranges where positioning and movement would be key to success. 

Now we will look at how many kills players who have entered the top 10 rankings accrue over the course of a game.

In [ ]:
tkill10 = PUBG_death_df[PUBG_death_df['killer_placement'] <= 10]

sns.countplot(tkill10, x='killer_placement')

As seen earlier first place players always have significantly more kills than 2nd place. Again emphasizing aggression as the ideal strategy

In [ ]:
sns.histplot(PUBG_death_df['time'], bins=50, kde=True)

We can further see this in this graph of when most kills happen in game which is near the beginning which further supports the aggressive playstyle to immediately thin out competitors.